# Carbon Engine — GPP vs NPP

Dual-panel animation comparing Gross and Net Primary Production
over Western Europe. Both products are 10-daily / dekadal at 300 m.


In [ ]:
from functools import partial
from pathlib import Path

from rs_tools.config import BoundingBox
from rs_tools.datasets.loader import load_dataset, load_passes_from_disk
from rs_tools.visualization.animation import save_timeseries_gif_lazy
from rs_tools.visualization.frames import make_dual_panel_composite
from rs_tools.visualization.clms_colormaps import CLMS_GPP, GPP_VMIN, GPP_VMAX, CLMS_NPP, NPP_VMIN, NPP_VMAX

In [ ]:
bbox = BoundingBox(west=-10, south=35, east=25, north=60)
DATA_DIR = "/home/bekaertd/RS_applications/Applications/CGOPS/carbon_engine"
gif_dir = Path('output/gifs')
gif_dir.mkdir(parents=True, exist_ok=True)

# Which dekads to include: [1], [2], [3], [1,2], etc.  None = all dekads.
DEKADS = None

## Load GPP and NPP dekads

In [ ]:
gpp_items = load_dataset(
    "CLMS_GPP_V2", bbox=bbox,
    start_date="2020-01-01", end_date="2026-03-01",
    limit=150, output_dir=f"{DATA_DIR}/gpp", dekads=DEKADS,
)
print(f"GPP: {len(gpp_items)} dekads")

In [ ]:
npp_items = load_dataset(
    "CLMS_NPP_V2", bbox=bbox,
    start_date="2020-01-01", end_date="2026-03-01",
    limit=150, output_dir=f"{DATA_DIR}/npp", dekads=DEKADS,
)
print(f"NPP: {len(npp_items)} dekads")

In [ ]:
# Reload as lightweight metadata references (no pixel data in RAM)
gpp_items = load_passes_from_disk(f"{DATA_DIR}/gpp", dekads=DEKADS)
npp_items = load_passes_from_disk(f"{DATA_DIR}/npp", dekads=DEKADS)
n = min(len(gpp_items), len(npp_items))
gpp_items, npp_items = gpp_items[:n], npp_items[:n]
print(f"GPP: {n} dekads  NPP: {n} dekads")

## Dual-panel GIF (lazy — one frame at a time)

In [ ]:
dual_composite = partial(
    make_dual_panel_composite,
    left_cmap=CLMS_GPP, right_cmap=CLMS_NPP,
    left_vmin=GPP_VMIN, left_vmax=GPP_VMAX,
    right_vmin=NPP_VMIN, right_vmax=NPP_VMAX,
    left_label="GPP", right_label="NPP",
)

def _composite(pair):
    left, right = pair
    return dual_composite(left, right)

gif_path = save_timeseries_gif_lazy(
    zip(gpp_items, npp_items),
    gif_dir / "carbon_engine_gpp_npp.gif",
    composite_fn=_composite,
    title="Carbon Engine — GPP vs NPP",
    fps=4,
    figsize=(16, 8),
)
print(f"Saved: {gif_path}")